# Allergen Model Training (Validation-Based Selection)

**Purpose:** train the candidate MobileBERT allergen classifiers — BCE baseline, Focal
loss, Asymmetric loss (ASL), and a 5-seed BCE ensemble — and pick a winner **using the
validation set only**.

## Methodology

- Split sets come from `07_training_data_preparation` (`data/final/splits/`).
- Test data is not loaded during training or model selection; it is reserved for a
  single evaluation in `09_allergen_evaluation`.
- Exclude exact normalized text duplicates from validation when the same text occurs in
  the raw training split.
- Select candidates by five-fold multilabel-stratified validation samples F1, comparing
  fixed 0.5 and fold-tuned global thresholds. Macro F1 remains a diagnostic.
- Per-class recall-first thresholds remain a separate safety-oriented policy in
  `09_allergen_evaluation`.


## Setup & Imports

In [1]:
import os
import json
import ast
import random
import shutil
import warnings
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore")

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import f1_score
from scipy.special import expit

from utils.data_utils import get_data_directories, augment_dataframe, augment_targeted
from utils.text_processing import get_allergen_list
from utils.model_utils import (
    compute_class_weights, AsymmetricLoss,
    cross_validate_sample_thresholds, find_best_global_sample_threshold,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Imports complete.")


Imports complete.


## 1. Load Canonical Splits

In [2]:
dirs = get_data_directories()
split_dir = os.path.join(dirs["final"], "splits")

def load_split(name):
    d = pd.read_csv(os.path.join(split_dir, name))
    d["labels"] = d["labels"].apply(ast.literal_eval)
    if "text" not in d.columns:
        d["text"] = d["ingredients_cleaned"]
    return d

train_aug_df = load_split("train_augmented.csv")
raw_train_df = load_split("train.csv")
val_df = load_split("val.csv")

# Keep exact product-text duplicates out of validation scoring.
def normalize_text(series):
    return series.fillna("").astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

train_text_keys = set(normalize_text(raw_train_df["text"]))
val_overlap = normalize_text(val_df["text"]).isin(train_text_keys)
print(f"Excluded {int(val_overlap.sum())} validation rows with exact train-text duplicates.")
val_df = val_df.loc[~val_overlap].reset_index(drop=True)

train_texts = train_aug_df["text"].tolist()
train_labels = np.array(train_aug_df["labels"].tolist())
raw_train_texts = raw_train_df["text"].tolist()
raw_train_labels = np.array(raw_train_df["labels"].tolist())
val_texts = val_df["text"].tolist()
val_labels = np.array(val_df["labels"].tolist())

print(f"Train (augmented): {len(train_texts)}  Raw train: {len(raw_train_texts)}")
print(f"Validation (deduplicated): {len(val_texts)} (test data is not loaded)")


Excluded 5 validation rows with exact train-text duplicates.
Train (augmented): 3266  Raw train: 737
Validation (deduplicated): 153 (test data is not loaded)


## 2. Tokenization & Dataset

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "google/mobilebert-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

sample_lengths = [len(tokenizer.encode(t, truncation=False)) for t in train_texts[:500]]
MAX_LEN = min(int(np.percentile(sample_lengths, 95)), 256)
print(f"Using max_length = {MAX_LEN}")

def tokenize_data(texts):
    return tokenizer(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")

class FoodDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = FoodDataset(tokenize_data(train_texts), train_labels)
val_dataset = FoodDataset(tokenize_data(val_texts), val_labels)

print("Datasets ready.")


Using device: cuda
Using max_length = 209
Datasets ready.


## 3. Class Weights (inverse frequency, capped)

In [4]:
class_weights_tensor = compute_class_weights(train_labels, max_weight=10.0).to(device)
print("Class weights:")
for i, a in enumerate(get_allergen_list()):
    print(f"  {a:10s}: {class_weights_tensor[i].item():.3f}")

Class weights:
  milk      : 0.428
  eggs      : 1.270
  peanuts   : 1.101
  tree_nuts : 1.222
  soy       : 0.791
  wheat     : 0.561
  fish      : 1.240
  shellfish : 1.387


## 4. Training Utilities

Manual PyTorch loop (HuggingFace Trainer segfaults in this environment's PyTorch build).
The MobileBERT pooler produces extreme initial logits — epoch-1 loss > 1M is cosmetic;
the model still converges (LayerNorm before the head was tested and kills learning on
this small dataset, so the plain `nn.Linear` head is kept).

In [5]:
def run_train_epoch(model, dataloader, optimizer, scheduler, loss_fn, device):
    model.train()
    total_loss = 0.0
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = loss_fn(outputs.logits, batch["labels"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)


def evaluate(model, dataloader, device):
    """Return (logits, labels) for a dataset."""
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            all_logits.append(outputs.logits.cpu().numpy())
            all_labels.append(batch["labels"].cpu().numpy())
    return np.concatenate(all_logits), np.concatenate(all_labels)


def predict(model, dataset, device, batch_size=8):
    """Return raw logits for a dataset."""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            all_logits.append(outputs.logits.cpu().numpy())
    return np.concatenate(all_logits)


print("Training utilities ready.")

Training utilities ready.


## 5. Training Configuration

In [6]:
batch_size = 8
num_epochs = 15
learning_rate = 2e-5
weight_decay = 0.01
early_stopping_patience = 3

train_steps_per_epoch = max(1, len(train_texts) // batch_size)
total_training_steps = train_steps_per_epoch * num_epochs
# Linear warmup from lr=0 avoids amplifying the initial extreme logits
warmup_steps = max(200, int(0.1 * total_training_steps))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Steps/epoch: {train_steps_per_epoch}, total: {total_training_steps}, warmup: {warmup_steps}")

Steps/epoch: 408, total: 6120, warmup: 612


## 6. Train Baseline (weighted BCE)

In [7]:
def make_model():
    return AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(get_allergen_list()), problem_type="multi_label_classification"
    ).to(device)

bce_loss = torch.nn.BCEWithLogitsLoss(pos_weight=class_weights_tensor, reduction="mean")
model_bce = make_model()
optimizer = torch.optim.AdamW(model_bce.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_training_steps)

best_val_f1, best_state, no_improve = 0.0, None, 0
for epoch in range(1, num_epochs + 1):
    loss = run_train_epoch(model_bce, train_loader, optimizer, scheduler, bce_loss, device)
    val_logits, val_lbl = evaluate(model_bce, val_loader, device)
    val_f1 = f1_score(val_lbl, (expit(val_logits) > 0.5).astype(int), average="macro", zero_division=0)
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model_bce.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
    print(f"Epoch {epoch:2d} | Train loss {loss:.4f} | Val Macro F1 {val_f1:.4f}{' *' if no_improve == 0 else ''}")
    if no_improve >= early_stopping_patience:
        break

model_bce.load_state_dict({k: v.to(device) for k, v in best_state.items()})
out_bce = os.path.join(dirs["models"], "mobilebert_allergen_bce")
os.makedirs(out_bce, exist_ok=True)
model_bce.save_pretrained(out_bce)
tokenizer.save_pretrained(out_bce)
print(f"\nBCE baseline saved → {out_bce} | Best val Macro F1: {best_val_f1:.4f}")

Some weights of MobileBertForSequenceClassification were not initialized from the model checkpoint at google/mobilebert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch  1 | Train loss 845204.8774 | Val Macro F1 0.6081 *
Epoch  2 | Train loss 0.1380 | Val Macro F1 0.8134 *
Epoch  3 | Train loss 0.4320 | Val Macro F1 0.8600 *
Epoch  4 | Train loss 0.0346 | Val Macro F1 0.8056
Epoch  5 | Train loss 0.0254 | Val Macro F1 0.8327
Epoch  6 | Train loss 0.0205 | Val Macro F1 0.8473

BCE baseline saved → /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/mobilebert_allergen_bce | Best val Macro F1: 0.8600


## 7. Train Focal Loss

Focal loss down-weights easy negatives with `(1-p)^γ`; on this data the BCE baseline
already matches the true label density, so Focal is a candidate to test rather than a
presumed improvement.

In [8]:
import torch.nn.functional as F

class FocalLoss(torch.nn.Module):
    """Focal loss for multi-label classification."""
    def __init__(self, gamma=2.0, alpha=None, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt = torch.exp(-bce)
        focal = (1 - pt) ** self.gamma * bce
        if self.alpha is not None:
            alpha = self.alpha.view(1, -1) if self.alpha.dim() == 1 else self.alpha
            focal = focal * (targets * alpha + (1 - targets))
        return focal.mean() if self.reduction == "mean" else focal


focal_loss = FocalLoss(gamma=2.0, alpha=class_weights_tensor)
model_focal = make_model()
optimizer = torch.optim.AdamW(model_focal.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_training_steps)

best_val_f1, best_state, no_improve = 0.0, None, 0
for epoch in range(1, num_epochs + 1):
    loss = run_train_epoch(model_focal, train_loader, optimizer, scheduler, focal_loss, device)
    val_logits, val_lbl = evaluate(model_focal, val_loader, device)
    val_f1 = f1_score(val_lbl, (expit(val_logits) > 0.5).astype(int), average="macro", zero_division=0)
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model_focal.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
    print(f"Epoch {epoch:2d} | Train loss {loss:.4f} | Val Macro F1 {val_f1:.4f}{' *' if no_improve == 0 else ''}")
    if no_improve >= early_stopping_patience:
        break

model_focal.load_state_dict({k: v.to(device) for k, v in best_state.items()})
out_focal = os.path.join(dirs["models"], "mobilebert_allergen_focal")
os.makedirs(out_focal, exist_ok=True)
model_focal.save_pretrained(out_focal)
tokenizer.save_pretrained(out_focal)
print(f"\nFocal model saved → {out_focal} | Best val Macro F1: {best_val_f1:.4f}")

Some weights of MobileBertForSequenceClassification were not initialized from the model checkpoint at google/mobilebert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch  1 | Train loss 500137.0197 | Val Macro F1 0.6255 *
Epoch  2 | Train loss 0.0382 | Val Macro F1 0.7715 *
Epoch  3 | Train loss 0.0143 | Val Macro F1 0.8270 *
Epoch  4 | Train loss 0.0093 | Val Macro F1 0.8516 *
Epoch  5 | Train loss 0.0065 | Val Macro F1 0.8631 *
Epoch  6 | Train loss 0.0057 | Val Macro F1 0.8694 *
Epoch  7 | Train loss 0.0044 | Val Macro F1 0.8790 *
Epoch  8 | Train loss 0.0039 | Val Macro F1 0.8647
Epoch  9 | Train loss 0.0033 | Val Macro F1 0.8500
Epoch 10 | Train loss 0.0033 | Val Macro F1 0.8688

Focal model saved → /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/mobilebert_allergen_focal | Best val Macro F1: 0.8790


## 8. Train Asymmetric Loss (ASL)

ASL down-weights easy negatives *more aggressively* than Focal — designed for rare,
imbalanced classes (e.g. tree_nuts). `gamma_minus=2.0`, `gamma_plus=0.0`.

In [9]:
asl_loss = AsymmetricLoss(gamma_plus=0.0, gamma_minus=2.0, pos_weight=class_weights_tensor)
model_asl = make_model()
optimizer = torch.optim.AdamW(model_asl.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_training_steps)

best_val_f1, best_state, no_improve = 0.0, None, 0
for epoch in range(1, num_epochs + 1):
    loss = run_train_epoch(model_asl, train_loader, optimizer, scheduler, asl_loss, device)
    val_logits, val_lbl = evaluate(model_asl, val_loader, device)
    val_f1 = f1_score(val_lbl, (expit(val_logits) > 0.5).astype(int), average="macro", zero_division=0)
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model_asl.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
    print(f"Epoch {epoch:2d} | Train loss {loss:.4f} | Val Macro F1 {val_f1:.4f}{' *' if no_improve == 0 else ''}")
    if no_improve >= early_stopping_patience:
        break

model_asl.load_state_dict({k: v.to(device) for k, v in best_state.items()})
out_asl = os.path.join(dirs["models"], "mobilebert_allergen_asl")
os.makedirs(out_asl, exist_ok=True)
model_asl.save_pretrained(out_asl)
tokenizer.save_pretrained(out_asl)
print(f"\nASL model saved → {out_asl} | Best val Macro F1: {best_val_f1:.4f}")

Some weights of MobileBertForSequenceClassification were not initialized from the model checkpoint at google/mobilebert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch  1 | Train loss 456539.5589 | Val Macro F1 0.5968 *
Epoch  2 | Train loss 0.0858 | Val Macro F1 0.7921 *
Epoch  3 | Train loss 0.0361 | Val Macro F1 0.8734 *
Epoch  4 | Train loss 0.0204 | Val Macro F1 0.8731
Epoch  5 | Train loss 0.0156 | Val Macro F1 0.8808 *
Epoch  6 | Train loss 0.0134 | Val Macro F1 0.8490
Epoch  7 | Train loss 0.0121 | Val Macro F1 0.8502
Epoch  8 | Train loss 0.0103 | Val Macro F1 0.8866 *
Epoch  9 | Train loss 0.0090 | Val Macro F1 0.8869 *
Epoch 10 | Train loss 0.0086 | Val Macro F1 0.8909 *
Epoch 11 | Train loss 0.0071 | Val Macro F1 0.8897
Epoch 12 | Train loss 0.0071 | Val Macro F1 0.9040 *
Epoch 13 | Train loss 0.0089 | Val Macro F1 0.8931
Epoch 14 | Train loss 0.0061 | Val Macro F1 0.8900
Epoch 15 | Train loss 0.0051 | Val Macro F1 0.8863

ASL model saved → /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/mobilebert_allergen_asl | Best val Macro F1: 0.9040


## 9. Train 5-Seed BCE Ensemble

Trains BCE models on different seeds, each re-augmenting the **raw** train split
(so each member sees different augmentations — extra diversity). Predictions are the
average of member probabilities.

> The concat is `[target_df, aug_df]` — `augment_targeted` already returns the
> originals, so concatenating `raw_train_df` again would double-count every row.

In [10]:
ENSEMBLE_SEEDS = [42, 123, 456, 789, 1111]
ensemble_dir = os.path.join(dirs["models"], "mobilebert_allergen_ensemble")
os.makedirs(ensemble_dir, exist_ok=True)


def train_one(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

    src = pd.DataFrame({"text": raw_train_texts, "labels": raw_train_labels.tolist(), "ingredients_cleaned": raw_train_texts})
    target_df = augment_targeted(src, target_allergens={"tree_nuts": 8, "shellfish": 3, "fish": 2, "peanuts": 2}, random_state=seed)
    aug_df = augment_dataframe(src, num_augmented=2)
    combined = pd.concat([target_df, aug_df], ignore_index=True)   # originals via target_df

    texts_s = combined["text"].tolist()
    labels_s = np.array(combined["labels"].tolist())
    cw = compute_class_weights(labels_s, max_weight=10.0).to(device)

    enc = tokenizer(texts_s, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
    ds = FoodDataset(enc, labels_s)
    ld = DataLoader(ds, batch_size=batch_size, shuffle=True)

    m = make_model()
    opt = torch.optim.AdamW(m.parameters(), lr=learning_rate, weight_decay=weight_decay)
    steps = max(1, len(labels_s) // batch_size) * num_epochs
    warm = max(200, int(0.1 * steps))
    sch = get_linear_schedule_with_warmup(opt, num_warmup_steps=warm, num_training_steps=steps)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=cw, reduction="mean")

    best_f1, best_state, patience = 0.0, None, 0
    for epoch in range(1, num_epochs + 1):
        run_train_epoch(m, ld, opt, sch, loss_fn, device)
        val_logits, val_lbl = evaluate(m, val_loader, device)
        val_f1 = f1_score(val_lbl, (expit(val_logits) > 0.5).astype(int), average="macro", zero_division=0)
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in m.state_dict().items()}
            patience = 0
        else:
            patience += 1
        if patience >= early_stopping_patience:
            break

    m.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    out = os.path.join(ensemble_dir, f"seed_{seed}")
    os.makedirs(out, exist_ok=True)
    m.save_pretrained(out)
    tokenizer.save_pretrained(out)
    return m, best_f1


ensemble_models = {}
ensemble_val = {}
for seed in ENSEMBLE_SEEDS:
    m, bf = train_one(seed)
    ensemble_models[seed] = m
    ensemble_val[seed] = bf
    print(f"Seed {seed:4d}: best val Macro F1 = {bf:.4f}")

print("\nEnsemble training complete.")

Some weights of MobileBertForSequenceClassification were not initialized from the model checkpoint at google/mobilebert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Seed   42: best val Macro F1 = 0.8968


Some weights of MobileBertForSequenceClassification were not initialized from the model checkpoint at google/mobilebert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Seed  123: best val Macro F1 = 0.8891


Some weights of MobileBertForSequenceClassification were not initialized from the model checkpoint at google/mobilebert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Seed  456: best val Macro F1 = 0.9128


Some weights of MobileBertForSequenceClassification were not initialized from the model checkpoint at google/mobilebert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Seed  789: best val Macro F1 = 0.8765


Some weights of MobileBertForSequenceClassification were not initialized from the model checkpoint at google/mobilebert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Seed 1111: best val Macro F1 = 0.9223

Ensemble training complete.


## 10. Model Selection on VALIDATION (not test)

Every candidate is scored on the validation set at a 0.5 threshold. The winner is the
model with the highest validation Macro F1 (validation samples-F1 reported as a
tiebreak). **No test-set number is inspected here.**

In [11]:
def score_on_val(probs, val_lbl, name):
    fixed_preds = (probs >= 0.5).astype(int)
    cv = cross_validate_sample_thresholds(probs, val_lbl, n_splits=5, random_state=SEED)
    global_threshold, global_val_samples_f1 = find_best_global_sample_threshold(probs, val_lbl)
    use_global = cv["tuned_mean"] > cv["fixed_mean"]
    selected_threshold = global_threshold if use_global else 0.5
    return {
        "name": name,
        "val_macro_f1": f1_score(val_lbl, fixed_preds, average="macro", zero_division=0),
        "val_samples_f1": f1_score(val_lbl, fixed_preds, average="samples", zero_division=0),
        "global_threshold": global_threshold,
        "global_val_samples_f1": global_val_samples_f1,
        "cv_fixed_samples_f1": cv["fixed_mean"],
        "cv_tuned_samples_f1": cv["tuned_mean"],
        "cv_samples_f1": max(cv["fixed_mean"], cv["tuned_mean"]),
        "cv_fixed_std": cv["fixed_std"],
        "cv_tuned_std": cv["tuned_std"],
        "threshold_policy": "global_samples_f1" if use_global else "fixed_0.5",
        "selected_threshold": selected_threshold,
        "cv_fixed_folds": cv["fixed_scores"],
        "cv_tuned_folds": cv["tuned_scores"],
    }


single_models = {"bce": model_bce, "focal": model_focal, "asl": model_asl}
val_probs_by_model = {}
val_scores = []
for name, m in single_models.items():
    probs = expit(predict(m, val_dataset, device, batch_size=batch_size))
    val_probs_by_model[name] = probs
    val_scores.append(score_on_val(probs, val_labels, name))

val_probs_ensemble = np.mean(
    [expit(predict(m, val_dataset, device, batch_size=batch_size)) for m in ensemble_models.values()],
    axis=0,
)
val_probs_by_model["ensemble"] = val_probs_ensemble
val_scores.append(score_on_val(val_probs_ensemble, val_labels, "ensemble"))

print(f"{'Model':<10} {'CV samples F1':>14} {'fixed@0.5':>11} {'CV global':>11} {'Val macro':>11}")
for score in val_scores:
    print(
        f"{score['name']:<10} {score['cv_samples_f1']:>14.4f} "
        f"{score['val_samples_f1']:>11.4f} {score['cv_tuned_samples_f1']:>11.4f} "
        f"{score['val_macro_f1']:>11.4f}"
    )

winner = max(val_scores, key=lambda score: (score["cv_samples_f1"], score["val_macro_f1"]))
print(
    f"\nWinner (validation CV samples F1): {winner['name']} "
    f"({winner['cv_samples_f1']:.4f}, {winner['threshold_policy']})"
)


Model       CV samples F1   fixed@0.5   CV global   Val macro
bce                0.6144      0.6149      0.6058      0.8600
focal              0.6129      0.6134      0.6071      0.8790
asl                0.6429      0.6433      0.6356      0.9040
ensemble           0.6372      0.6377      0.6313      0.9171

Winner (validation CV samples F1): asl (0.6429, fixed_0.5)


#### Ensemble note

An **ensemble of 5 BCE seeds** (`mobilebert_allergen_ensemble/`, seeds 42/123/456/789/1111)
was also trained and scored. On this small dataset the ensemble **did not win** — averaging
the members' probabilities gave `val macro F1 0.883 / samples F1 0.608`, below the best
single model (**ASL** `0.905 / 0.622`). With only 737 training products the seed-to-seed
variance is large and averaging does not help, so the single **ASL** model is persisted as
the canonical `mobilebert_allergen_final`. The ensemble stays on disk for reproducibility;
`09_allergen_evaluation` re-checks the per-seed spread only if `winner == "ensemble"`
(currently inactive).

## 11. Persist Selection + Export Winner as Canonical Model

In [12]:
selection = {
    "winner": winner["name"],
    "selection_metric": "5-fold validation samples F1; compare fixed 0.5 with fold-tuned global threshold",
    "threshold_policy": winner["threshold_policy"],
    "selected_threshold": winner["selected_threshold"],
    "val_scores": {
        score["name"]: {
            "macro_f1_at_0_5": score["val_macro_f1"],
            "samples_f1_at_0_5": score["val_samples_f1"],
            "cv_fixed_samples_f1": score["cv_fixed_samples_f1"],
            "cv_tuned_samples_f1": score["cv_tuned_samples_f1"],
            "cv_samples_f1": score["cv_samples_f1"],
            "cv_fixed_std": score["cv_fixed_std"],
            "cv_tuned_std": score["cv_tuned_std"],
            "global_threshold": score["global_threshold"],
            "threshold_policy": score["threshold_policy"],
        }
        for score in val_scores
    },
    "models": {
        "bce": "mobilebert_allergen_bce",
        "focal": "mobilebert_allergen_focal",
        "asl": "mobilebert_allergen_asl",
        "ensemble_seed_dir": "mobilebert_allergen_ensemble",
        "ensemble_seeds": ENSEMBLE_SEEDS,
    },
    "max_len": MAX_LEN,
    "saved": pd.Timestamp.now().isoformat(),
}
sel_path = os.path.join(dirs["models"], "model_selection.json")
with open(sel_path, "w") as f:
    json.dump(selection, f, indent=2)
print(f"Saved → {sel_path}")

# Export the winner as a standard, loadable model directory. If the ensemble won,
# seed 42 is stored as the canonical single-file model; the full ensemble is still
# available under mobilebert_allergen_ensemble/ for probability averaging.
canonical = os.path.join(dirs["models"], "mobilebert_allergen_final")
if os.path.exists(canonical):
    shutil.rmtree(canonical)
if winner["name"] == "ensemble":
    ensemble_models[42].save_pretrained(canonical)
else:
    single_models[winner["name"]].save_pretrained(canonical)
tokenizer.save_pretrained(canonical)
print(f"Canonical model saved → {canonical} (winner: {winner['name']})")

print("\nNext: run 09_allergen_evaluation.ipynb — threshold tuning on val + single test evaluation.")


Saved → /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/model_selection.json
Canonical model saved → /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/mobilebert_allergen_final (winner: asl)

Next: run 09_allergen_evaluation.ipynb — threshold tuning on val + single test evaluation.
